# 05 — Bike Utilisation & Retirement Risk
**Question:** Which bikes are over-worked and potentially unsafe due to missing maintenance records?

Key finding: Bike #12942 — 8,197 rentals, 7.7 years, 2,810 hours ridden, zero workshop records.

In [ ]:
from sqlalchemy import create_engine
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

PROJECT = 'm2-dataset-study'
engine = create_engine(f'bigquery://{PROJECT}/london_bicycles')
plt.rcParams['figure.dpi'] = 120

## 5.1 — Distribution of total rentals per bike

In [ ]:
sql = """
SELECT
    bike_id,
    total_rentals,
    total_hours_ridden,
    avg_rentals_per_day,
    active_lifespan_days,
    has_workshop_record,
    exceeds_retirement_threshold,
    is_high_risk_no_maintenance,
    model_category
FROM `m2-dataset-study.london_bicycles.mart_bike_utilisation`
WHERE bike_id IS NOT NULL
"""
df = pd.read_sql(sql, engine)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rental count distribution
ax = axes[0]
ax.hist(df['total_rentals'], bins=80, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(5000, color='red', linestyle='--', linewidth=1.5, label='Retirement threshold (5k)')
ax.set_xlabel('Total Rentals per Bike')
ax.set_ylabel('Number of Bikes')
ax.set_title('Distribution of Rentals per Bike', fontsize=12, fontweight='bold')
ax.legend()

# Hours ridden distribution
ax2 = axes[1]
ax2.hist(df['total_hours_ridden'].dropna(), bins=80, color='darkorange', edgecolor='white', alpha=0.8)
ax2.axvline(2000, color='red', linestyle='--', linewidth=1.5, label='Retirement threshold (2k hrs)')
ax2.set_xlabel('Total Hours Ridden')
ax2.set_ylabel('Number of Bikes')
ax2.set_title('Distribution of Hours Ridden per Bike', fontsize=12, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.savefig('../outputs/05_utilisation_distribution.png', bbox_inches='tight')
plt.show()

print(f"Total bikes: {len(df):,}")
print(f"Exceed retirement threshold: {df['exceeds_retirement_threshold'].sum():,}")
print(f"HIGH RISK (threshold + no maintenance record): {df['is_high_risk_no_maintenance'].sum():,}")

## 5.2 — Top 15 hardest-working bikes

In [ ]:
top15 = df.nlargest(15, 'total_rentals')[[
    'bike_id','total_rentals','total_hours_ridden',
    'avg_rentals_per_day','active_lifespan_days',
    'has_workshop_record','is_high_risk_no_maintenance'
]].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 6))
colours = ['#E74C3C' if risk else '#2980B9' for risk in top15['is_high_risk_no_maintenance']]
bars = ax.barh(top15['bike_id'].astype(str), top15['total_rentals'], color=colours)
ax.set_xlabel('Total Rentals')
ax.set_title('Top 15 Highest-Utilisation Bikes\nRed = exceeds threshold with NO maintenance record',
             fontsize=12, fontweight='bold')

for bar, row in zip(bars, top15.itertuples()):
    label = f"{row.total_rentals:,} | {row.total_hours_ridden:.0f} hrs | {row.avg_rentals_per_day}/day"
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=8)

ax.set_xlim(right=top15['total_rentals'].max() * 1.3)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#E74C3C', label='High risk: over threshold, no maintenance record'),
    Patch(color='#2980B9', label='Over threshold, workshop record exists'),
], fontsize=10)
plt.tight_layout()
plt.savefig('../outputs/05_top_bikes.png', bbox_inches='tight')
plt.show()
print(top15.to_string(index=False))

## 5.3 — Maintenance trip analysis

In [ ]:
maint_sql = """
SELECT
    workshop_name,
    maintenance_duration_bucket,
    COUNT(*) AS trips,
    ROUND(AVG(duration_hours), 1) AS avg_hours,
    ROUND(MAX(duration_days), 1) AS max_days
FROM `m2-dataset-study.london_bicycles.mart_maintenance_trips`
GROUP BY workshop_name, maintenance_duration_bucket
ORDER BY workshop_name, trips DESC
"""
maint = pd.read_sql(maint_sql, engine)
print("Maintenance Trip Summary:")
print(maint.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
pivot = maint.groupby(['workshop_name','maintenance_duration_bucket'])['trips'].sum().unstack(fill_value=0)
pivot.plot(kind='bar', ax=ax, colormap='Set2')
ax.set_title('Maintenance Trips by Workshop and Duration', fontsize=12, fontweight='bold')
ax.set_ylabel('Trip Count')
ax.set_xlabel('')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('../outputs/05_maintenance_breakdown.png', bbox_inches='tight')
plt.show()